# **SWIFT: PhysioSpecNet Deepfake Telephony Detection**
### **Academic Journal Benchmark Pipeline (IEEE ICASSP / Interspeech / IEEE TIFS)**
---
This notebook trains **PhysioSpecNet** on genuine multi-speaker audio with telephonic codec degradation (G.711 $\mu$-law / PSTN bandpass), calibrated probabilities, and an automated 3-way ablation study:
1. **Baseline**: 1-Channel LFCC + EfficientNet-B0
2. **Ablation**: 6-Channel Direct Stack (No Cross-Attention)
3. **Proposed**: 6-Channel PhysioSpecNet with Cross-Channel Attention

Outputs publication-ready **300 DPI figures**, **calibrated minDCF (0.01 - 0.25)**, **Equal Error Rate (EER)**, and **LaTeX table code**.

## **Step 1: Verify GPU Acceleration**

In [ ]:
!nvidia-smi

## **Step 2: Clone SWIFT Repository & Install Dependencies**

In [ ]:
# Clean existing if re-running
%cd /content
!rm -rf SWIFT
!git clone https://github.com/codebreaker77/SWIFT.git
%cd SWIFT

# Install dependencies
!pip install -q scipy scikit-learn matplotlib datasets soundfile onnxruntime

## **Step 3 (Recommended): Ingest Real Speech Datasets**
*(Downloads genuine multi-speaker real speech audio from LibriSpeech clean partitions to ensure the model learns vocal tract biomechanics rather than procedural artifacts)*

In [ ]:
# Fetch genuine multi-speaker dataset samples directly into data/seed_audio
!python -c "from src.fetch_hf_dataset_samples import fetch_and_export_hf_samples; fetch_and_export_hf_samples()"

# Check count of seed waveforms
!find public/samples -name "*.wav" | wc -l

## **Step 4: Run Calibrated Academic Training & Ablation Study**
*(Applies Platt probability calibration to fix minDCF posterior collapse, weight decay $10^{-4}$, and G.711 $\mu$-law transcoding)*

In [ ]:
!python scripts/train_colab.py --samples 2500 --epochs 25 --batch_size 32 --out_dir paper_metrics_colab

## **Step 5: View Generated 300 DPI Publication Figures**

In [ ]:
from IPython.display import Image, display

print("--- Figure 1: Convergence & Equal Error Rate Trajectories ---")
display(Image("paper_metrics_colab/fig1_convergence_ablation.png"))

print("--- Figure 2: ROC Curve Comparison (AUC Benchmarks) ---")
display(Image("paper_metrics_colab/fig2_roc_curves_comparison.png"))

## **Step 6: View Formatted LaTeX Table for Journal Submission**

In [ ]:
with open("paper_metrics_colab/table_journal_ablation.tex", "r") as f:
    print(f.read())

## **Step 7: Download Publication Artifacts & Trained Checkpoint**

In [ ]:
from google.colab import files

# Zip artifacts for 1-click download
!zip -r swift_paper_results.zip paper_metrics_colab checkpoints/best_physiospecnet.pth
files.download("swift_paper_results.zip")